In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# 1. LOAD DATA
print("Loading train.csv and test.csv...")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

# 2. ADVANCED FEATURE ENGINEERING
print("Performing feature engineering...")
for df in [train, test]:
    # Body Composition
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    # Draft history signal: use train mapping to avoid KeyError for test
    if 'Drafted' in df.columns:
        df['Position_Draft_Rate'] = df.groupby('Position')['Drafted'].transform('mean')
    else:
        pos_rate = train.groupby('Position')['Drafted'].mean()
        df['Position_Draft_Rate'] = df['Position'].map(pos_rate).fillna(0)

    # Combine rank within position
    df['Speed_Rank'] = df.groupby('Position')['Sprint_40yd'].rank(ascending=True)
    df['Jump_Rank'] = df.groupby('Position')['Vertical_Jump'].rank(ascending=False)
    df['Combined_Rank'] = df['Speed_Rank'] + df['Jump_Rank']

    # Interaction between size and agility
    # Size-Adjusted Agility (use components directly to avoid referencing Total_Agility before it's created)
    df['Size_Agility'] = df['BMI'] * (df['Agility_3cone'] + df['Shuttle'])
    # Agility ratio
    df['Agility_Ratio'] = df['Agility_3cone'] / df['Shuttle']

    # Strength per unit weight
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']

    # Speed relative to position type
    df['Speed_pos_type_diff'] = df['Sprint_40yd'] - df.groupby('Position_Type')['Sprint_40yd'].transform('mean')
    
    # Size-Adjusted Athleticism
    df['Speed_Score'] = df['Weight'] / df['Sprint_40yd']
    
    # Power & Explosiveness
    # Combining Vertical and Broad jump captures lower-body power better than one alone
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump'] 
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    
    # Directional Change
    # Agility_3cone and Shuttle are highly correlated (0.88)
    df['Total_Agility'] = df['Agility_3cone'] + df['Shuttle']
    
    # Missing Value Indicators
    # Knowing a test was skipped can be a predictor in itself
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
    
    # Composite Metrics
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump']/100) + (df['Broad_Jump']/100) - (df['Shuttle']/10)
    df['Height_Adj_Speed'] = (df['Weight'] * (df['Height'] ** 0.5)) / (df['Sprint_40yd'] ** 4)
    
    # Position-Relative Stats
    for stat in ['Sprint_40yd', 'BMI', 'Catch_Radius']:
        df[f'{stat}_pos_diff'] = df[stat] - df.groupby('Position')[stat].transform('mean')

# 3. PREPROCESSING
print("Preprocessing features...")
# Handle 'School' using Frequency Encoding instead of dropping it
school_freq = train['School'].value_counts().to_dict()
train['School_Freq'] = train['School'].map(school_freq)
test['School_Freq'] = test['School'].map(school_freq).fillna(0)

# Drop high-cardinality School name and Id
train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])

# Label Encoding for remaining categoricals
le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Advanced Imputation using IterativeImputer (MICE)
X = train.drop(columns=["Drafted"])
y = train["Drafted"]
imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_imputed = pd.DataFrame(imputer.transform(test), columns=test.columns)

# 4. CUSTOM NEURAL NETWORK
class DraftNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

def train_draftnet(X_train, y_train, input_dim, epochs=80, batch_size=64, device='cpu'):
    model = DraftNet(input_dim).to(device)
    criterion = nn.BCELoss()
    # Using AdamW with standard weight decay
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    X_tensor = torch.FloatTensor(X_train).to(device)
    y_tensor = torch.FloatTensor(y_train).view(-1, 1).to(device)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    # drop_last=True prevents BatchNorm1d from crashing if the final batch contains only 1 sample
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model

def predict_draftnet(model, X_val, device='cpu'):
    model.eval()
    X_tensor = torch.FloatTensor(X_val).to(device)
    with torch.no_grad():
        preds = model(X_tensor).cpu().numpy().flatten()
    return preds

# 5. DEFINE BASE MODELS
cat_model = CatBoostClassifier(
    iterations=250, depth=8, learning_rate=0.05,
    subsample=0.9, bootstrap_type='Bernoulli',
    random_seed=2025, verbose=0
)
xgb_model = XGBClassifier(
    n_estimators=150, max_depth=5, learning_rate=0.05,
    random_state=2025, eval_metric='logloss'
)
lgbm_model = LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    random_state=2025, objective='binary', verbosity=-1
)

rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=42)

# Level 1 - Out-Of-Fold (OOF) predictions
oof_cat  = np.zeros(len(X_imputed))
oof_xgb  = np.zeros(len(X_imputed))
oof_lgbm = np.zeros(len(X_imputed))
oof_nn   = np.zeros(len(X_imputed))

# Test predictions
test_cat  = np.zeros(len(test_imputed))
test_xgb  = np.zeros(len(test_imputed))
test_lgbm = np.zeros(len(test_imputed))
test_nn   = np.zeros(len(test_imputed))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected for Neural Network training: {device}")

# 6. TRAINING LOOP (CROSS-VALIDATION)
print("Starting cross-validation stacking training loop (10 folds total)...")
for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_imputed, y)):
    X_t, X_v = X_imputed.iloc[train_idx], X_imputed.iloc[valid_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[valid_idx]
    
    # Fit scaling specifically for the Neural Network to prevent data leakage
    scaler = StandardScaler()
    X_t_scaled = scaler.fit_transform(X_t)
    X_v_scaled = scaler.transform(X_v)
    test_imputed_scaled = scaler.transform(test_imputed)
    
    # --- Model 1: CatBoost ---
    cat_model.fit(X_t, y_t)
    oof_cat[valid_idx]  = cat_model.predict_proba(X_v)[:, 1]
    test_cat           += cat_model.predict_proba(test_imputed)[:, 1] / 10
    
    # --- Model 2: XGBoost ---
    xgb_model.fit(X_t, y_t)
    oof_xgb[valid_idx]  = xgb_model.predict_proba(X_v)[:, 1]
    test_xgb           += xgb_model.predict_proba(test_imputed)[:, 1] / 10
    
    # --- Model 3: LightGBM ---
    lgbm_model.fit(X_t, y_t)
    oof_lgbm[valid_idx] = lgbm_model.predict_proba(X_v)[:, 1]
    test_lgbm          += lgbm_model.predict_proba(test_imputed)[:, 1] / 10
    
    # --- Model 4: Custom PyTorch DraftNet ---
    nn_model = train_draftnet(X_t_scaled, y_t.values, input_dim=X_t.shape[1], epochs=80, batch_size=64, device=device)
    oof_nn[valid_idx]   = predict_draftnet(nn_model, X_v_scaled, device=device)
    test_nn            += predict_draftnet(nn_model, test_imputed_scaled, device=device) / 10
    
    if (fold + 1) % 10 == 0:
        print(f"  Completed fold {fold + 1}/10...")



Loading train.csv and test.csv...
Performing feature engineering...
Preprocessing features...
Device selected for Neural Network training: cpu
Starting cross-validation stacking training loop (10 folds total)...


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
# 1. LOAD DATA
print("Loading train.csv and test.csv...")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")
# 2. ADVANCED FEATURE ENGINEERING
print("Performing feature engineering...")
for df in [train, test]:
    # Body Composition
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    # Draft history signal: use train mapping to avoid KeyError for test
    if 'Drafted' in df.columns:
        df['Position_Draft_Rate'] = df.groupby('Position')['Drafted'].transform('mean')
    else:
        pos_rate = train.groupby('Position')['Drafted'].mean()
        df['Position_Draft_Rate'] = df['Position'].map(pos_rate).fillna(0)
    # Combine rank within position
    df['Speed_Rank'] = df.groupby('Position')['Sprint_40yd'].rank(ascending=True)
    df['Jump_Rank'] = df.groupby('Position')['Vertical_Jump'].rank(ascending=False)
    df['Combined_Rank'] = df['Speed_Rank'] + df['Jump_Rank']
    # Interaction between size and agility
    # Size-Adjusted Agility (use components directly to avoid referencing Total_Agility before it's created)
    df['Size_Agility'] = df['BMI'] * (df['Agility_3cone'] + df['Shuttle'])
    # Agility ratio
    df['Agility_Ratio'] = df['Agility_3cone'] / df['Shuttle']
    # Strength per unit weight
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']
    # Speed relative to position type
    df['Speed_pos_type_diff'] = df['Sprint_40yd'] - df.groupby('Position_Type')['Sprint_40yd'].transform('mean')
    
    # Size-Adjusted Athleticism
    df['Speed_Score'] = df['Weight'] / df['Sprint_40yd']
    
    # Power & Explosiveness
    # Combining Vertical and Broad jump captures lower-body power better than one alone
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump'] 
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    
    # Directional Change
    # Agility_3cone and Shuttle are highly correlated (0.88)
    df['Total_Agility'] = df['Agility_3cone'] + df['Shuttle']
    
    # Missing Value Indicators
    # Knowing a test was skipped can be a predictor in itself
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
    
    # Composite Metrics
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump']/100) + (df['Broad_Jump']/100) - (df['Shuttle']/10)
    df['Height_Adj_Speed'] = (df['Weight'] * (df['Height'] ** 0.5)) / (df['Sprint_40yd'] ** 4)
    
    # Position-Relative Stats
    for stat in ['Sprint_40yd', 'BMI', 'Catch_Radius']:
        df[f'{stat}_pos_diff'] = df[stat] - df.groupby('Position')[stat].transform('mean')
# 3. PREPROCESSING
print("Preprocessing features...")
# Handle 'School' using Frequency Encoding instead of dropping it
school_freq = train['School'].value_counts().to_dict()
train['School_Freq'] = train['School'].map(school_freq)
test['School_Freq'] = test['School'].map(school_freq).fillna(0)
# Drop high-cardinality School name and Id
train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])
# Label Encoding for remaining categoricals
le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
# Advanced Imputation using IterativeImputer (MICE)
X = train.drop(columns=["Drafted"])
y = train["Drafted"]
imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_imputed = pd.DataFrame(imputer.transform(test), columns=test.columns)
# 4. CUSTOM NEURAL NETWORK
class DraftNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)
def train_draftnet(X_train, y_train, input_dim, epochs=80, batch_size=64, device='cpu'):
    model = DraftNet(input_dim).to(device)
    criterion = nn.BCELoss()
    # Using AdamW with standard weight decay
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    
    X_tensor = torch.FloatTensor(X_train).to(device)
    y_tensor = torch.FloatTensor(y_train).view(-1, 1).to(device)
    
    dataset = TensorDataset(X_tensor, y_tensor)
    # drop_last=True prevents BatchNorm1d from crashing if the final batch contains only 1 sample
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
    return model
def predict_draftnet(model, X_val, device='cpu'):
    model.eval()
    X_tensor = torch.FloatTensor(X_val).to(device)
    with torch.no_grad():
        preds = model(X_tensor).cpu().numpy().flatten()
    return preds
# 5. DEFINE BASE MODELS
cat_model = CatBoostClassifier(
    iterations=250, depth=8, learning_rate=0.05,
    subsample=0.9, bootstrap_type='Bernoulli',
    random_seed=2025, verbose=0
)
xgb_model = XGBClassifier(
    n_estimators=150, max_depth=5, learning_rate=0.05,
    random_state=2025, eval_metric='logloss'
)
lgbm_model = LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    random_state=2025, objective='binary', verbosity=-1
)
rskf = RepeatedStratifiedKFold(n_splits=20, n_repeats=3, random_state=42)
# Level 1 - Out-Of-Fold (OOF) predictions
oof_cat  = np.zeros(len(X_imputed))
oof_xgb  = np.zeros(len(X_imputed))
oof_lgbm = np.zeros(len(X_imputed))
oof_nn   = np.zeros(len(X_imputed))
# Test predictions
test_cat  = np.zeros(len(test_imputed))
test_xgb  = np.zeros(len(test_imputed))
test_lgbm = np.zeros(len(test_imputed))
test_nn   = np.zeros(len(test_imputed))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected for Neural Network training: {device}")
# 6. TRAINING LOOP (CROSS-VALIDATION)
print("Starting cross-validation stacking training loop (60 folds total)...")
for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_imputed, y)):
    X_t, X_v = X_imputed.iloc[train_idx], X_imputed.iloc[valid_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[valid_idx]
    
    # Fit scaling specifically for the Neural Network to prevent data leakage
    scaler = StandardScaler()
    X_t_scaled = scaler.fit_transform(X_t)
    X_v_scaled = scaler.transform(X_v)
    test_imputed_scaled = scaler.transform(test_imputed)
    
    # --- Model 1: CatBoost ---
    cat_model.fit(X_t, y_t)
    oof_cat[valid_idx]  = cat_model.predict_proba(X_v)[:, 1]
    test_cat           += cat_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    
    # --- Model 2: XGBoost ---
    xgb_model.fit(X_t, y_t)
    oof_xgb[valid_idx]  = xgb_model.predict_proba(X_v)[:, 1]
    test_xgb           += xgb_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    
    # --- Model 3: LightGBM ---
    lgbm_model.fit(X_t, y_t)
    oof_lgbm[valid_idx] = lgbm_model.predict_proba(X_v)[:, 1]
    test_lgbm          += lgbm_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    
    # --- Model 4: Custom PyTorch DraftNet ---
    nn_model = train_draftnet(X_t_scaled, y_t.values, input_dim=X_t.shape[1], epochs=80, batch_size=64, device=device)
    oof_nn[valid_idx]   = predict_draftnet(nn_model, X_v_scaled, device=device)
    test_nn            += predict_draftnet(nn_model, test_imputed_scaled, device=device) / (20 * 3)
    
    if (fold + 1) % 10 == 0:
        print(f"  Completed fold {fold + 1}/60...")
# 7. EVALUATE BASE MODEL PERFORMANCES
print("\n--- Out-of-Fold (OOF) CV AUC Scores ---")
auc_cat = roc_auc_score(y, oof_cat)
auc_xgb = roc_auc_score(y, oof_xgb)
auc_lgbm = roc_auc_score(y, oof_lgbm)
auc_nn = roc_auc_score(y, oof_nn)
print(f"CatBoost OOF AUC: {auc_cat:.5f}")
print(f"XGBoost OOF AUC:  {auc_xgb:.5f}")
print(f"LightGBM OOF AUC: {auc_lgbm:.5f}")
print(f"DraftNet OOF AUC: {auc_nn:.5f}")
# 8. STACKED META LEARNER
print("\nTraining Meta-Learner (Logistic Regression)...")
meta_train = np.column_stack([oof_cat, oof_xgb, oof_lgbm, oof_nn])
meta_test  = np.column_stack([test_cat, test_xgb, test_lgbm, test_nn])
meta_model = LogisticRegression()
meta_model.fit(meta_train, y)
# Meta predictions & AUC
final_oof_preds = meta_model.predict_proba(meta_train)[:, 1]
auc_meta = roc_auc_score(y, final_oof_preds)
print(f"Stacked Meta-Model OOF AUC: {auc_meta:.5f}")
# Meta learner coefficients
coefs = meta_model.coef_[0]
print("\nMeta-Learner Coefficients:")
print(f"  CatBoost weight: {coefs[0]:.4f}")
print(f"  XGBoost weight:  {coefs[1]:.4f}")
print(f"  LightGBM weight: {coefs[2]:.4f}")
print(f"  DraftNet weight: {coefs[3]:.4f}")
# 9. GENERATE SUBMISSION
print("\nGenerating final submission file...")
final_test_preds = meta_model.predict_proba(meta_test)[:, 1]
submission = sample_sub.copy()
submission["Drafted"] = final_test_preds
submission.to_csv("submission_stacking.csv", index=False)
print("Saved submission to 'submission_stacking.csv'. Stacking training complete!")

In [17]:
# Skip the training loop and set manual coefficients directly
# Create dummy OOF arrays (or load from file if you have previous runs)
oof_cat  = np.zeros(len(X_imputed))
oof_xgb  = np.zeros(len(X_imputed))
oof_lgbm = np.zeros(len(X_imputed))
oof_nn   = np.zeros(len(X_imputed))

test_cat  = np.zeros(len(test_imputed))
test_xgb  = np.zeros(len(test_imputed))
test_lgbm = np.zeros(len(test_imputed))
test_nn   = np.zeros(len(test_imputed))

# Now manually set the meta-learner coefficients
meta_train = np.column_stack([oof_cat, oof_xgb, oof_lgbm, oof_nn])
meta_test  = np.column_stack([test_cat, test_xgb, test_lgbm, test_nn])

meta_model = LogisticRegression()
# Manually set your weights here
meta_model.coef_ = np.array([[0.25, 0.35, 0.25, 0.15]])  # [CatBoost, XGBoost, LightGBM, DraftNet]
meta_model.intercept_ = np.array([0.0])

print("\nMeta-Learner Coefficients (manually set):")
coefs = meta_model.coef_[0]
print(f"  CatBoost weight: {coefs[0]:.4f}")
print(f"  XGBoost weight:  {coefs[1]:.4f}")
print(f"  LightGBM weight: {coefs[2]:.4f}")
print(f"  DraftNet weight: {coefs[3]:.4f}")


Meta-Learner Coefficients (manually set):
  CatBoost weight: 0.2500
  XGBoost weight:  0.3500
  LightGBM weight: 0.2500
  DraftNet weight: 0.1500
